# cooprecsys — Collaborative Filtering Tutorial
**Repository:** https://github.com/masterofray/cooprecsys

End-to-end walkthrough: pseudo-rating → ALS training → evaluation → candidate generation.

In [ ]:
# Install dependencies
# !pip install cooprecsys implicit duckdb mlflow tqdm faiss-cpu -q

## 1. Simulate Implicit Interaction Data

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

np.random.seed(42)
N_USERS, N_ITEMS, N_INTERACTIONS = 10_000, 5_000, 200_000

interactions = pd.DataFrame({
    'user_id':       np.random.randint(0, N_USERS, N_INTERACTIONS).astype(str),
    'item_id':       np.random.randint(0, N_ITEMS, N_INTERACTIONS).astype(str),
    'click_count':   np.random.poisson(1.5, N_INTERACTIONS),
    'view_count':    np.random.poisson(3.0, N_INTERACTIONS),
    'purchase_count':np.random.binomial(1, 0.1, N_INTERACTIONS),
    'dwell_seconds': np.random.exponential(60, N_INTERACTIONS),
    'timestamp':     [datetime(2024,1,1) + timedelta(seconds=int(s))
                      for s in np.random.randint(0, 86400*365, N_INTERACTIONS)]
})
print(f'Interactions shape: {interactions.shape}')
interactions.head(3)

## 2. Build Pseudo-Ratings with CounterFeitCore

In [ ]:
from cooprecsys.preprocessing.counterfeit_core import CounterFeitCore

engine = CounterFeitCore(method='weighted')
rated_df = engine.build_pseudo_ratings(interactions)
print(f'Rating range: {rated_df.pseudo_rating.min():.2f} – {rated_df.pseudo_rating.max():.2f}')
rated_df.describe()

## 3. Build Sparse Interaction Matrix

In [ ]:
from cooprecsys.preprocessing.cf_preprocess import build_interaction_matrix, temporal_train_test_split

train_df, test_df = temporal_train_test_split(rated_df, timestamp_col=None)
# For demo: random split since no timestamp in rated_df
idx = int(len(rated_df) * 0.8)
train_df, test_df = rated_df.iloc[:idx], rated_df.iloc[idx:]

train_mat, user_enc, item_enc = build_interaction_matrix(train_df)
print(f'Train matrix: {train_mat.shape} | nnz={train_mat.nnz:,}')

## 4. Train AryColBring (ALS)

In [ ]:
from cooprecsys.models.trainer import AryColBringTrainer

trainer = AryColBringTrainer(
    n_factors=64, n_iterations=15, regularization=0.01, alpha=40.0
)
trainer.fit(train_mat, show_progress=True)

## 5. Offline Evaluation

In [ ]:
test_mat, _, _ = build_interaction_matrix(test_df)
metrics = trainer.evaluate(test_mat, k_values=[5, 10, 20], max_users=500)
import pandas as pd
pd.Series(metrics).to_frame('value').round(4)

## 6. Generate Recommendations for a Single User

In [ ]:
from cooprecsys.models.predictor import AryColBringPredictor

predictor = AryColBringPredictor(trainer, user_enc, item_enc)
recs = predictor.recommend(user_id_raw='42', top_k=10)
pd.DataFrame(recs)

## 7. Key Takeaways
- `CounterFeitCore` converts raw implicit signals to pseudo-ratings via DuckDB CTEs.
- `AryColBringTrainer` wraps ALS with RAM-aware batch inference.
- Offline NDCG@10 and online CTR are both tracked via MLflow.
- Cold-start users are routed through a content-based fallback layer.
